# Fine-tuning a model with the Trainer API

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
# !pip install datasets evaluate transformers[sentencepiece]

The code examples below assume we have already executed the examples in the previous section. Here is a short summary recapping what we need:

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

### Training

The first step before we can define our Trainer is to define a TrainingArguments class that will contain all the hyperparameters the Trainer will use for training and evaluation. The only argument we have to provide is a directory where the trained model will be saved, as well as the checkpoints along the way. For all the rest, we can leave the defaults, which should work pretty well for a basic fine-tuning.

In [3]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer")

The second step is to define our model. As in the previous chapter, we will use the `AutoModelForSequenceClassification` class, with two labels:

In [4]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Unlike in Chapter 2, we get a **warning** after instantiating this pretrained model. This is because BERT has not been pretrained on classifying pairs of sentences, so the head of the pretrained model has been discarded and a new head suitable for sequence classification has been added instead. The warnings indicate that some weights were not used (the ones corresponding to the dropped pretraining head) and that some others were randomly initialized (the ones for the new head). It concludes by encouraging us to train the model, which is exactly what we are going to do now.


Once we have our model, we can define a `Trainer` by passing it all the objects constructed up to now — the model, the training_args, the training and validation datasets, our `data_collator`, and our `processing_class`. The `processing_class` parameter is a newer addition that tells the Trainer which tokenizer to use for processing:

In [5]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

When we pass a tokenizer as the `processing_class`, the default `data_collator` used by the Trainer will be a `DataCollatorWithPadding`. We could have skipped the `data_collator=data_collator` line in this case, but we included it here to show this important part of the processing pipeline.

In [6]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
CUDA version: 13.0
GPU count: 1
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


To fine-tune the model on our dataset, we just have to call the `train()` method of our `Trainer`:

In [7]:
trainer.train()

Step,Training Loss
500,0.612300
1000,0.457400


TrainOutput(global_step=1377, training_loss=0.4790563590299763, metrics={'train_runtime': 135.3246, 'train_samples_per_second': 81.316, 'train_steps_per_second': 10.176, 'total_flos': 405114969714960.0, 'train_loss': 0.4790563590299763, 'epoch': 3.0})

The above code do not tell how well (or badly) the model is performing. This is because:

1. We didn’t tell the `Trainer` to evaluate during training by setting `eval_strategy` in `TrainingArguments` to either "steps" (evaluate every `eval_steps`) or "epoch" (evaluate at the end of each epoch).
2. We didn’t provide the `Trainer` with a `compute_metrics()` function to calculate a metric during said evaluation (otherwise the evaluation would just have printed the loss, which is not a very intuitive number).

### Evaluation

Let’s see how we can build a useful `compute_metrics()` function and use it the next time we train. The function must take an `EvalPrediction` object (which is a named tuple with a predictions field and a label_ids field) and will return a dictionary mapping strings to floats (the strings being the names of the metrics returned, and the floats their values). To get some predictions from our model, we can use the `Trainer.predict()` command:

In [8]:
predictions = trainer.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)

(408, 2) (408,)


The output of the `predict()` method is another named tuple with three fields: `predictions`, `label_ids`, and `metrics`. The `metrics` field will just contain the loss on the dataset passed, as well as some time metrics (how long it took to predict, in total and on average). Once we complete our `compute_metrics()` function and pass it to the `Trainer`, that field will also contain the metrics returned by `compute_metrics()`.

As we can see, `predictions` is a two-dimensional array with shape 408 x 2 (408 being the number of elements in the dataset we used). Those are the logits for each element of the dataset we passed to `predict()` (as we saw in the previous chapter, all Transformer models return logits). To transform them into predictions that we can compare to our labels, we need to take the index with the maximum value on the second axis:

In [9]:
import numpy as np

preds = np.argmax(predictions.predictions, axis=-1)

We can now compare those `preds` to the labels. To build our `compute_metric()` function, we will rely on the metrics from the **🤗 Evaluate** library. We can load the metrics associated with the MRPC dataset as easily as we loaded the dataset, this time with the `evaluate.load()` function. The object returned has a `compute()` method we can use to do the metric calculation:

In [10]:
import evaluate

metric = evaluate.load("glue", "mrpc")
metric.compute(predictions=preds, references=predictions.label_ids)

{'accuracy': 0.821078431372549, 'f1': 0.8768971332209107}

Wrapping everything together, we get our compute_metrics() function:

In [11]:
def compute_metrics(eval_preds):
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

And to see it used in action to report metrics at the end of each epoch, here is how we define a new `Trainer` with this `compute_metrics()` function:

In [12]:
training_args = TrainingArguments(
    "test-trainer", eval_strategy="epoch"
)  # main article has 'evaluation_strategy', but in recent versions of Transformers, 'evaluation_strategy' was renamed to 'eval_strategy'.
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Note that we create a new `TrainingArguments` with its `eval_strategy` set to "epoch" and a new model — otherwise, we would just be continuing the training of the model we have already trained. To launch a new training run, we execute:

In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.430748,0.835784,0.880570
2,0.559400,0.443663,0.845588,0.891566
3,0.330200,0.676904,0.852941,0.898990


TrainOutput(global_step=1377, training_loss=0.37937139373978757, metrics={'train_runtime': 165.9086, 'train_samples_per_second': 66.326, 'train_steps_per_second': 8.3, 'total_flos': 405114969714960.0, 'train_loss': 0.37937139373978757, 'epoch': 3.0})

### Advanced Training Features

The `Trainer` comes with many built-in features that make modern deep learning best practices accessible:

#### Mixed Precision Training: 
Use `fp16=True` in training arguments for faster training and reduced memory usage:

```Python
training_args = TrainingArguments(
    "test-trainer",
    eval_strategy="epoch",
    fp16=True,  # Enable mixed precision
)
```

In [14]:
training_args = TrainingArguments(
    "test-trainer", eval_strategy="epoch", fp16=True
)  # main article has 'evaluation_strategy', but in recent versions of Transformers, 'evaluation_strategy' was renamed to 'eval_strategy'.
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.592263,0.703431,0.815267
2,0.629000,0.492520,0.776961,0.854400
3,0.570400,0.481177,0.803922,0.867987


TrainOutput(global_step=1377, training_loss=0.5650512429364106, metrics={'train_runtime': 168.2976, 'train_samples_per_second': 65.384, 'train_steps_per_second': 8.182, 'total_flos': 405114969714960.0, 'train_loss': 0.5650512429364106, 'epoch': 3.0})

#### Gradient Accumulation: 
For effective larger batch sizes when GPU memory is limited:

```Python
training_args = TrainingArguments(
    "test-trainer",
    eval_strategy="epoch",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch size = 4 * 4 = 16
)
```

#### Learning Rate Scheduling: 
The Trainer uses linear decay by default, but we can customize this:

```Python
training_args = TrainingArguments(
    "test-trainer",
    eval_strategy="epoch",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",  # Try different schedulers
)
```